# RAG with MiniMax and LanceDB

This notebook demonstrates how to build a Retrieval-Augmented Generation (RAG) pipeline using:
- **[sentence-transformers](https://www.sbert.net/)** for generating text embeddings
- **[LanceDB](https://lancedb.com/)** for vector storage and similarity search
- **[MiniMax M2.7](https://www.minimaxi.com/)** as the LLM for answer generation (via OpenAI-compatible API)

MiniMax provides an OpenAI-compatible endpoint at `https://api.minimax.io/v1`, so we can use the standard `openai` Python SDK.

## Install dependencies

In [ ]:
!pip install lancedb sentence-transformers openai beautifulsoup4 requests

## Import libraries

In [ ]:
import os
import re

import lancedb
import pandas as pd
import requests
from bs4 import BeautifulSoup
from openai import OpenAI
from sentence_transformers import SentenceTransformer

## Set your MiniMax API key

Get your API key from [MiniMax Platform](https://www.minimaxi.com/).

In [ ]:
os.environ["MINIMAX_API_KEY"] = ""  # Set your MiniMax API key here

## Scrape web content

In [ ]:
def scrape_content(url: str) -> str:
    """Scrape text content from a web page."""
    try:
        response = requests.get(url, timeout=15)
        soup = BeautifulSoup(response.content, "html.parser")
        paragraphs = soup.find_all("p")
        return " ".join(para.get_text() for para in paragraphs)
    except Exception as e:
        print(f"Failed to scrape {url}: {e}")
        return ""

In [ ]:
urls = [
    "https://lancedb.github.io/lancedb/",
    "https://lancedb.github.io/lancedb/basic/",
    "https://lancedb.github.io/lancedb/concepts/vector_search/",
]

documents = {}
for url in urls:
    content = scrape_content(url)
    if content:
        documents[url] = content
        print(f"Scraped {url}: {len(content)} chars")
    else:
        print(f"Skipped {url}")

## Chunk text and generate embeddings

In [ ]:
def chunk_text(text: str, chunk_size: int = 500) -> list:
    """Split text into fixed-size chunks by sentences."""
    sentences = re.split(r"(?<=[.!?])\s+", text)
    chunks, current = [], ""
    for sentence in sentences:
        if len(current) + len(sentence) > chunk_size and current:
            chunks.append(current.strip())
            current = sentence
        else:
            current = f"{current} {sentence}" if current else sentence
    if current.strip():
        chunks.append(current.strip())
    return chunks

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

all_chunks = []
all_embeddings = []

for url, content in documents.items():
    chunks = chunk_text(content)
    for chunk in chunks:
        if len(chunk.strip()) < 20:
            continue
        embedding = model.encode(chunk)
        all_chunks.append(chunk)
        all_embeddings.append(embedding)

print(f"Total chunks: {len(all_chunks)}")

## Store in LanceDB

In [ ]:
df = pd.DataFrame({"vector": all_embeddings, "text": all_chunks})

db = lancedb.connect("lancedb_minimax")
table = db.create_table("rag_docs", data=df, mode="overwrite")

print(f"Stored {len(df)} chunks in LanceDB")
table.to_pandas().head()

## Retrieve relevant documents

In [ ]:
query = "What is LanceDB and how does vector search work?"

query_embedding = model.encode(query)
results = table.search(query_embedding).limit(5).to_pandas()

print(f"Top {len(results)} results for: '{query}'\n")
for i, row in results.iterrows():
    print(f"--- Chunk {i+1} (distance: {row['_distance']:.4f}) ---")
    print(row["text"][:200] + "...\n")

## Generate answer with MiniMax M2.7

MiniMax provides an OpenAI-compatible API at `https://api.minimax.io/v1`.  
We use the standard `openai` SDK with a custom `base_url`.

In [ ]:
client = OpenAI(
    api_key=os.environ.get("MINIMAX_API_KEY", ""),
    base_url="https://api.minimax.io/v1",
)

context = "\n\n---\n\n".join(list(results["text"]))

response = client.chat.completions.create(
    model="MiniMax-M2.7",
    messages=[
        {
            "role": "system",
            "content": (
                "You are a helpful assistant. Answer the user's question based on "
                "the provided context. If the context does not contain enough "
                "information, say so clearly."
            ),
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {query}",
        },
    ],
    temperature=0.7,
    max_tokens=1024,
)

print("MiniMax M2.7 Answer:")
print(response.choices[0].message.content)

## Try another query

In [ ]:
query2 = "How do I create a table and add data in LanceDB?"

query_embedding2 = model.encode(query2)
results2 = table.search(query_embedding2).limit(5).to_pandas()
context2 = "\n\n---\n\n".join(list(results2["text"]))

response2 = client.chat.completions.create(
    model="MiniMax-M2.7",
    messages=[
        {
            "role": "system",
            "content": (
                "You are a helpful assistant. Answer the user's question based on "
                "the provided context. If the context does not contain enough "
                "information, say so clearly."
            ),
        },
        {
            "role": "user",
            "content": f"Context:\n{context2}\n\nQuestion: {query2}",
        },
    ],
    temperature=0.7,
    max_tokens=1024,
)

print(f"Query: {query2}\n")
print("MiniMax M2.7 Answer:")
print(response2.choices[0].message.content)